# 🛡️ Glu-Stock: 02_SIGNAL_INFERENCE
**Phase**: Ensemble Intelligence (RF + CNN)

This notebook retrieves candidates from the Firebase `research` queue, performs dual-brain inference, and pushes high-conviction signals to the `signals` queue.

In [ ]:
!pip install -q yfinance firebase-admin pandas scikit-learn joblib tensorflow python-dotenv ta xgboost lightgbm catboost

In [ ]:
# 🏗️ SECTION 2: INFRASTRUCTURE (Firebase & Secrets)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite
from firebase_admin import credentials, firestore
from datetime import datetime

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try: tg = user_secrets.get_secret("TELEGRAM_TOKEN")
            except: tg = None
            return {
                "key": json.loads(user_secrets.get_secret("FIREBASE_KEY_JSON")),
                "telegram": tg
            }
        else:
            from dotenv import load_dotenv
            load_dotenv()
            return {
                "key": json.loads(os.getenv("FIREBASE_KEY_JSON", "{}")),
                "telegram": os.getenv("TELEGRAM_TOKEN")
            }

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
        
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
        
    def push_task(self, queue_name: str, data):
        # Wrap in payload to allow lists
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
        
    def insert_trade(self, trade_data):
        self.db.collection("glu_stock_trades").add(trade_data)
        
    def get_history(self, limit=5):
        docs = self.db.collection("glu_stock_history").order_by("timestamp", direction=firestore.Query.DESCENDING).limit(limit).get()
        history = [doc.to_dict() for doc in docs]
        return {str(i): h for i, h in enumerate(reversed(history))} if history else {}
        
    def get_active_trades(self):
        docs = self.db.collection("glu_stock_trades").where("status", "==", "OPEN").get()
        return [doc.to_dict() for doc in docs]
        
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})

def get_dynamic_lq45():
    print("🌐 Fetching latest LQ45 constituents...")
    fallback = ["ACES.JK", "ADRO.JK", "AKRA.JK", "AMMN.JK", "AMRT.JK", "ANTM.JK", "ARTO.JK", "ASII.JK", "BBCA.JK", "BBNI.JK", "BBRI.JK", "BBTN.JK", "BMRI.JK", "BRIS.JK", "BRPT.JK", "BUKA.JK", "CPIN.JK", "CTRA.JK", "ESSA.JK", "EXCL.JK", "GGRM.JK", "GOTO.JK", "HRUM.JK", "ICBP.JK", "INCO.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK", "ITMG.JK", "KLBF.JK", "MAPI.JK", "MBMA.JK", "MDKA.JK", "MEDC.JK", "MTEL.JK", "PGAS.JK", "PGEO.JK", "PTBA.JK", "SIDO.JK", "SMGR.JK", "SRTG.JK", "TLKM.JK", "TPIA.JK", "UNTR.JK"]
    try:
        tables = pd.read_html('https://id.wikipedia.org/wiki/LQ45')
        for df in tables:
            if 'Kode' in df.columns:
                return (df['Kode'] + '.JK').tolist()
            elif 'Ticker' in df.columns:
                return (df['Ticker'] + '.JK').tolist()
    except:
        pass
    return fallback


In [ ]:
# 🧠 SECTION 3: CORE LOGIC (Institutional Predictors & Meta-Label Gate)
import ta

def frac_diff(series, d=0.4, thres=1e-5):
    weights = [1.0]
    for k in range(1, len(series)):
        w = -weights[-1] * (d - k + 1) / k
        if abs(w) < thres:
            break
        weights.append(w)
    weights = np.array(weights[::-1])
    result = np.full(len(series), np.nan)
    for t in range(len(weights) - 1, len(series)):
        result[t] = np.dot(weights, series[t - len(weights) + 1:t + 1])
    return result

class MLPredictor:
    def __init__(self, model_dir):
        path = os.path.join(model_dir, 'glu_brain_v1.joblib')
        brain = joblib.load(path)
        self.model = brain.get('model')
        self.features = brain.get('features', [])
        self.train_acc = brain.get('train_accuracy', 0)
        self.val_acc = brain.get('val_accuracy', 0)
        print(f'✅ RF Model loaded: {len(self.features)} features | Val Acc: {self.val_acc:.2%}')

    def build_features(self, df):
        close = df['Close'].squeeze()
        high = df['High'].squeeze()
        low = df['Low'].squeeze()
        volume = df['Volume'].squeeze()
        feat = pd.DataFrame(index=df.index)
        feat['Returns'] = close.pct_change()
        feat['RSI'] = ta.momentum.RSIIndicator(close=close, window=14).rsi()
        macd = ta.trend.MACD(close=close)
        feat['MACD'] = macd.macd_diff()
        boll = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
        feat['BB_High'] = boll.bollinger_hband_indicator()
        feat['BB_Low'] = boll.bollinger_lband_indicator()
        feat['ATR'] = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
        feat['ADX'] = ta.trend.ADXIndicator(high=high, low=low, close=close, window=14).adx()
        obv = ta.volume.OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()
        feat['OBV_norm'] = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-7)
        feat['day_of_week'] = df.index.dayofweek
        feat['week_of_month'] = (df.index.day - 1) // 7
        fd = frac_diff(close.values, d=0.4)
        feat['frac_diff_close'] = fd
        vol_ma = volume.rolling(20).mean()
        feat['vol_ratio'] = volume / (vol_ma + 1e-7)
        feat = feat.dropna()
        return feat

    def predict(self, df):
        try:
            feat = self.build_features(df)
            if len(feat) == 0:
                return 0, 0.5
            row = feat[self.features].tail(1)
            pred = self.model.predict(row)[0]
            proba = self.model.predict_proba(row)[0]
            confidence = max(proba)
            return int(pred), float(confidence)
        except Exception:
            return 0, 0.5

class CNNPredictor:
    def __init__(self, model_dir):
        self.interpreters = {}
        for h in ['daily_t2']:
            path = os.path.join(model_dir, f'cnn_{h}.tflite')
            if os.path.exists(path):
                self.interpreters[h] = tflite.Interpreter(model_path=path)
                self.interpreters[h].allocate_tensors()
                print(f'✅ CNN TFLite loaded: {h}')

    def predict(self, df):
        try:
            interpreter = self.interpreters['daily_t2']
            close = df['Close'].squeeze()
            high = df['High'].squeeze()
            low = df['Low'].squeeze()
            volume = df['Volume'].squeeze()
            atr = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
            raw = np.column_stack([df['Open'].squeeze().values, high.values, low.values, close.values, volume.values, atr.values])
            valid_start = np.argmax(~np.isnan(raw).any(axis=1))
            raw = raw[valid_start:]
            seq = raw[-30:]
            seq_min = seq.min(axis=0)
            seq_max = seq.max(axis=0)
            norm_seq = (seq - seq_min) / (seq_max - seq_min + 1e-7)
            input_details = interpreter.get_input_details()
            expected_shape = input_details[0]['shape']
            n_features = expected_shape[-1]
            input_data = np.expand_dims(norm_seq[:, :n_features].astype(np.float32), axis=0)
            interpreter.set_tensor(input_details[0]['index'], input_data)
            interpreter.invoke()
            output = interpreter.get_tensor(interpreter.get_output_details()[0]['index'])[0]
            return float(output[1]) if len(output) > 1 else float(output[0])
        except Exception as e:
            print(f'⚠️ CNN predict error: {e}')
            return 0.5


In [ ]:
# 🚀 SECTION 4: MAIN EXECUTION (Meta-Label Gated)
META_CONFIDENCE_THRESHOLD = 0.60

def run_inference():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    
    model_dir = '/kaggle/input/glustock-brains'
    if not os.path.exists(model_dir):
        print('❌ Models missing! Please add glustock-brains dataset.')
        return
    
    candidates = fb.get_and_clear_queue('research')
    if not candidates:
        print('📭 Queue empty.')
        return
    
    rf = MLPredictor(model_dir)
    cnn = CNNPredictor(model_dir)
    signals = {}
    skipped = 0
    
    for ticker_list in candidates:
        for ticker in ticker_list:
            print(f'🔬 Analyzing {ticker}...')
            df = yf.download(ticker, period='90d', interval='1d', progress=False)
            if len(df) < 50:
                continue
            
            # 1. RF Base Prediction (3-class: -1, 0, +1)
            rf_signal, rf_conf = rf.predict(df)
            
            # 2. CNN Meta-Label Confidence
            cnn_conf = cnn.predict(df)
            
            # 3. Meta-Label Gate: Only execute if RF says BUY (+1) AND CNN confidence is high
            if rf_signal == 1 and cnn_conf >= META_CONFIDENCE_THRESHOLD:
                signals[ticker] = {
                    'conviction': float(cnn_conf),
                    'rf_signal': int(rf_signal),
                    'rf_confidence': float(rf_conf),
                    'meta_confidence': float(cnn_conf),
                    'price': float(df['Close'].iloc[-1]),
                    'timestamp': datetime.now().isoformat()
                }
                print(f'🔥 {ticker} APPROVED (RF: +1 @ {rf_conf:.0%} | Meta: {cnn_conf:.0%})')
            else:
                skipped += 1
                reason = 'RF=HOLD/SELL' if rf_signal != 1 else f'Meta={cnn_conf:.0%}<{META_CONFIDENCE_THRESHOLD:.0%}'
                print(f'⛔ {ticker} BLOCKED ({reason})')
    
    if signals:
        fb.push_task('signals', signals)
        log_lines = [
            f'Meta-Label Inference Complete.',
            f'✅ Approved: {len(signals)} signals',
            f'⛔ Blocked: {skipped} (False Positive filter)',
            f'🎯 Gate Threshold: {META_CONFIDENCE_THRESHOLD:.0%}',
        ]
        for t, s in signals.items():
            log_lines.append(f'  {t}: RF={s["rf_confidence"]:.0%} | Meta={s["meta_confidence"]:.0%}')
        fb.log_event('INFERENCE', chr(10).join(log_lines))
        for line in log_lines:
            print(line)
    else:
        fb.log_event('INFERENCE', f'No signals passed Meta-Label gate. {skipped} candidates blocked.')
        print(f'🛡️ All {skipped} candidates blocked by Meta-Label gate.')

run_inference()
